# ML-Guided Paired CDR3 Library Design

This notebook uses a **pairwise Ridge regression model** trained on the double-mutant DMS data
to derive improved per-(position, amino-acid) scores for IUPAC codon selection.

## Why this improves on the heuristic design

The original `abDMS_CDR3_tcraft_paired_opool_design.ipynb` computes a *position-level* synergy bonus:
it asks "do alpha position *i* and beta position *j* tend to interact positively?" — averaged over all
amino acid identities at those positions.

This notebook instead fits:
```
log2fc(α_i=a, β_j=b) = w_α(i,a) + w_β(j,b) + w_αβ(i,a,j,b)
```
learning **amino-acid-specific** interchain interactions.
The marginal score for `(chain, pos, aa)` is then the average predicted binding over enriched partner
amino acids — a more accurate input to IUPAC codon selection.

## Caveats
- The model improves **interchain** epistasis resolution; it cannot improve intrachain extrapolation
  (multi-site sequences within one chain still assume additivity — same as the heuristic design).
- If cross-validated R² is < 0.25 for a peptide, the model has not learned much beyond the additive
  baseline and the marginals will closely resemble the original single-mutant scores.
- This is **one-shot model-guided design**, not iterative active learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from collections import defaultdict
from itertools import product
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score as _cv_score

NB_DIR   = Path('/cluster/project/reddy/marluca/NGS_pipeline/notebooks')
DATA_DIR = Path('/cluster/project/reddy/marluca/NGS_pipeline/data/P3620_LUCA-TCRDMF5-abDMS/04_variant_labeling')
OUT_DIR  = NB_DIR / 'opool_designs_tcraft_ml'
OUT_DIR.mkdir(exist_ok=True)
print(f'Output: {OUT_DIR}')

In [ ]:
PEPTIDES = ['DRG01','DRG04','GIG01','GIG02','GIG03','GIG04','GIG05']

WT_ALPHA  = 'AVNFGGGKLI'
WT_BETA   = 'ASSLSFGTEAF'
ALPHA_LEN = len(WT_ALPHA)
BETA_LEN  = len(WT_BETA)

ENRICHMENT_COL = {p: 'enrich_pos1x_vs_neg' for p in PEPTIDES}
ENRICHMENT_COL['GIG02'] = 'enrich_pos2x_vs_neg'
PSEUDOCOUNT = 1e-3

# ── Design parameters (match original notebook) ───────────────────────────
TARGET_DIVERSITY  = 10_000_000
MAX_DIVERSITY     = 15_000_000
MIN_SCORE         = 0.3
K_MAX             = 8
SYN_WEIGHT        = 0.5      # kept for reference; not used by ML score fn
TIER1_FRAC        = 0.33
STOP_PENALTY      = 10.0
EFFICIENCY_THRESH = 0.3
TOP_K             = 0

TIER1_EXCLUDE = {'alpha': {0}, 'beta': {0, 1}}

print(f'Alpha WT: {WT_ALPHA}  ({ALPHA_LEN} aa)')
print(f'Beta  WT: {WT_BETA}  ({BETA_LEN} aa)')
print(f'{len(PEPTIDES)} peptides: {PEPTIDES}')

## Genetic Code and IUPAC Utilities

Copied verbatim from the original design notebook.

In [ ]:
GENETIC_CODE = {
    'ATA':'I','ATC':'I','ATT':'I','ATG':'M',
    'ACA':'T','ACC':'T','ACG':'T','ACT':'T',
    'AAC':'N','AAT':'N','AAA':'K','AAG':'K',
    'AGC':'S','AGT':'S','AGA':'R','AGG':'R',
    'CTA':'L','CTC':'L','CTG':'L','CTT':'L',
    'CCA':'P','CCC':'P','CCG':'P','CCT':'P',
    'CAC':'H','CAT':'H','CAA':'Q','CAG':'Q',
    'CGA':'R','CGC':'R','CGG':'R','CGT':'R',
    'GTA':'V','GTC':'V','GTG':'V','GTT':'V',
    'GCA':'A','GCC':'A','GCG':'A','GCT':'A',
    'GAC':'D','GAT':'D','GAA':'E','GAG':'E',
    'GGA':'G','GGC':'G','GGG':'G','GGT':'G',
    'TCA':'S','TCC':'S','TCG':'S','TCT':'S',
    'TTC':'F','TTT':'F','TTA':'L','TTG':'L',
    'TAC':'Y','TAT':'Y','TAA':'*','TAG':'*',
    'TGC':'C','TGT':'C','TGA':'*','TGG':'W',
}
IUPAC_BASES = {
    'A':{'A'},'C':{'C'},'G':{'G'},'T':{'T'},
    'R':{'A','G'},'Y':{'C','T'},'S':{'G','C'},'W':{'A','T'},
    'K':{'G','T'},'M':{'A','C'},
    'B':{'C','G','T'},'D':{'A','G','T'},'H':{'A','C','T'},'V':{'A','C','G'},
    'N':{'A','C','G','T'},
}

def decode_iupac(triplet):
    codons = [b1+b2+b3 for b1 in IUPAC_BASES[triplet[0]]
                        for b2 in IUPAC_BASES[triplet[1]]
                        for b3 in IUPAC_BASES[triplet[2]]]
    aas, stops = set(), 0
    for c in codons:
        aa = GENETIC_CODE.get(c,'?')
        if aa == '*': stops += 1
        else: aas.add(aa)
    return aas, stops/len(codons)

ALL_IUPAC = list(IUPAC_BASES.keys())
IUPAC_PROPS = {}
for _t in product(ALL_IUPAC, repeat=3):
    _triplet = ''.join(_t)
    _aas, _sf = decode_iupac(_triplet)
    if _aas:
        IUPAC_PROPS[_triplet] = {'aas': frozenset(_aas), 'stop_frac': _sf}

print(f'Pre-computed {len(IUPAC_PROPS)} valid IUPAC triplets')

## Load Data: Single-Mutant Effects and Paired Epistasis

In [ ]:
def load_single_effects(pep_list):
    result = {}
    for pep in pep_list:
        df  = pd.read_csv(DATA_DIR / f'{pep}.variant_labeling.csv')
        col = ENRICHMENT_COL[pep]
        if col not in df.columns: col = 'enrich_pos1x_vs_neg'
        df['log2e']     = np.log2(df[col].clip(lower=PSEUDOCOUNT))
        baseline        = df['log2e'].mean()
        df['log2e_dev'] = df['log2e'] - baseline
        rows = []
        for _, r in df.iterrows():
            v = r['log2e_dev']
            aseq, bseq = r['alpha_aa'], r['beta_aa']
            if aseq != WT_ALPHA:
                for pos in range(ALPHA_LEN):
                    if aseq[pos] != WT_ALPHA[pos]:
                        rows.append(('alpha', pos, aseq[pos], v)); break
            if bseq != WT_BETA:
                for pos in range(BETA_LEN):
                    if bseq[pos] != WT_BETA[pos]:
                        rows.append(('beta', pos, bseq[pos], v)); break
        fx = (pd.DataFrame(rows, columns=['chain','position','aa','log2e_dev'])
              .groupby(['chain','position','aa'])['log2e_dev']
              .agg(log2fc='mean', n_variants='count').reset_index())
        result[pep] = fx
    return result

print('Loading single-mutant effects...')
SINGLE_FX = load_single_effects(PEPTIDES)
print(f'Done. DRG01: {len(SINGLE_FX["DRG01"])} entries')
SINGLE_FX['DRG01'].head(4)

In [ ]:
def compute_epistasis(pep_list):
    result = {}
    for pep in pep_list:
        df  = pd.read_csv(DATA_DIR / f'{pep}.variant_labeling.csv')
        col = ENRICHMENT_COL[pep]
        if col not in df.columns: col = 'enrich_pos1x_vs_neg'
        df['log2e']     = np.log2(df[col].clip(lower=PSEUDOCOUNT))
        baseline        = df['log2e'].mean()
        df['log2e_dev'] = df['log2e'] - baseline
        fx = SINGLE_FX[pep].set_index(['chain','position','aa'])['log2fc'].to_dict()
        rows = []
        for _, r in df.iterrows():
            aseq, bseq = r['alpha_aa'], r['beta_aa']
            if aseq == WT_ALPHA or bseq == WT_BETA: continue
            apos = aaa = bpos = baa = None
            for pos in range(ALPHA_LEN):
                if aseq[pos] != WT_ALPHA[pos]: apos, aaa = pos, aseq[pos]; break
            for pos in range(BETA_LEN):
                if bseq[pos] != WT_BETA[pos]: bpos, baa = pos, bseq[pos]; break
            if apos is None or bpos is None: continue
            ma = fx.get(('alpha', apos, aaa), 0.0)
            mb = fx.get(('beta',  bpos, baa), 0.0)
            obs = r['log2e_dev']
            rows.append({'alpha_pos': apos, 'aa_alpha': aaa,
                         'beta_pos':  bpos, 'aa_beta':  baa,
                         'log2fc_obs': obs, 'log2fc_exp': ma+mb,
                         'epistasis': obs - (ma+mb)})
        result[pep] = pd.DataFrame(rows)
    return result

print('Computing paired epistasis...')
EPISTASIS = compute_epistasis(PEPTIDES)
print(f'Done. DRG01: {len(EPISTASIS["DRG01"]):,} pairs')
EPISTASIS['DRG01'].head(4)

## ML Model: Pairwise Ridge Regression

For each peptide, fit a regularised linear model on the ~30K–120K double-mutant observations:

```
log2fc(α_i=a, β_j=b) = w_α(i,a) + w_β(j,b) + w_αβ(i,a,j,b)
```

| Feature block | Size | Description |
|---|---|---|
| α main effects | 10×20 = 200 | one weight per (alpha_pos, aa) |
| β main effects | 11×20 = 220 | one weight per (beta_pos, aa) |
| Interchain interactions | 200×220 = 44,000 | one weight per (α_pos,α_aa, β_pos,β_aa) pair |

L2 regularisation (Ridge) shrinks uncertain interaction weights toward zero — critical because
many of the 44,000 pairs are unobserved or low-coverage in the DMS.

In [ ]:
AAS_ORDERED = list('ACDEFGHIKLMNPQRSTVWY')
_aa_ord     = {aa: i for i, aa in enumerate(AAS_ORDERED)}
N_AA        = len(AAS_ORDERED)

N_ALPHA_FEAT = ALPHA_LEN * N_AA          # 200
N_BETA_FEAT  = BETA_LEN  * N_AA          # 220
N_INTERACT   = N_ALPHA_FEAT * N_BETA_FEAT  # 44,000
N_FEAT_TOTAL = N_ALPHA_FEAT + N_BETA_FEAT + N_INTERACT  # 44,420


def _build_X(apos, aai, bpos, bai):
    """Vectorised sparse feature matrix: alpha main + beta main + interchain interactions."""
    n  = len(apos)
    af = apos * N_AA + aai                    # alpha feature index in [0, N_ALPHA_FEAT)
    bf = bpos * N_AA + bai                    # beta  feature index in [0, N_BETA_FEAT)
    rows = np.tile(np.arange(n), 3)
    cols = np.concatenate([
        af,
        N_ALPHA_FEAT + bf,
        N_ALPHA_FEAT + N_BETA_FEAT + af * N_BETA_FEAT + bf,
    ])
    return sp.csr_matrix((np.ones(3 * n, np.float32), (rows, cols)),
                         shape=(n, N_FEAT_TOTAL))


def _epistasis_to_X(df):
    """Feature matrix + target vector from an EPISTASIS DataFrame."""
    valid = df['aa_alpha'].isin(_aa_ord) & df['aa_beta'].isin(_aa_ord)
    df = df[valid]
    X = _build_X(
        df['alpha_pos'].values.astype(int),
        df['aa_alpha'].map(_aa_ord).values.astype(int),
        df['beta_pos'].values.astype(int),
        df['aa_beta'].map(_aa_ord).values.astype(int),
    )
    return X, df['log2fc_obs'].values


def _grid_X():
    """Feature matrix for the full 44,000-combination prediction grid."""
    apos = np.repeat(np.arange(ALPHA_LEN), N_AA * BETA_LEN * N_AA)
    aai  = np.tile(np.repeat(np.arange(N_AA), BETA_LEN * N_AA), ALPHA_LEN)
    bpos = np.tile(np.repeat(np.arange(BETA_LEN), N_AA), ALPHA_LEN * N_AA)
    bai  = np.tile(np.arange(N_AA), ALPHA_LEN * N_AA * BETA_LEN)
    X    = _build_X(apos, aai, bpos, bai)
    pairs = pd.DataFrame({
        'alpha_pos': apos, 'aa_alpha': np.array(AAS_ORDERED)[aai],
        'beta_pos':  bpos, 'aa_beta':  np.array(AAS_ORDERED)[bai],
    })
    return X, pairs


# Pre-compute once — reused for every peptide
_X_GRID, _PAIRS_GRID = _grid_X()
print(f'Prediction grid: {len(_PAIRS_GRID):,} (α_pos, α_aa, β_pos, β_aa) combinations')
print(f'Feature matrix:  {N_FEAT_TOTAL:,} features  '
      f'({N_ALPHA_FEAT} α-main + {N_BETA_FEAT} β-main + {N_INTERACT:,} interactions)')

In [ ]:
# ── Fit one Ridge model per peptide ──────────────────────────────────────
ML_MODELS = {}
ML_R2     = {}

print(f'{"Peptide":<8}  {"n_pairs":>9}  {"α (Ridge)":>10}  {"CV R²":>8}')
print('─' * 44)
for pep in PEPTIDES:
    X, y  = _epistasis_to_X(EPISTASIS[pep].dropna(subset=['log2fc_obs']))
    model = RidgeCV(alphas=[0.1, 1, 10, 100, 1000], cv=5)
    model.fit(X, y)
    r2    = _cv_score(RidgeCV(alphas=[model.alpha_]), X, y, cv=5, scoring='r2').mean()
    ML_MODELS[pep] = model
    ML_R2[pep]     = r2
    print(f'{pep:<8}  {len(y):>9,}  {model.alpha_:>10.1f}  {r2:>8.3f}')

print('\nR² ≥ 0.3 → meaningful interchain signal beyond the additive baseline.')
print('R² < 0.1 → model will fall back toward single-mutant additive scores.')

## Compute ML Marginal Scores

For IUPAC codon selection we need a single score per `(chain, pos, aa)`.

**Alpha marginal(i, a)** = mean predicted log2fc over all beta (pos, aa) combinations,
weighted by clipped-positive beta single-mutant log2fc.

**Beta marginal(j, b)** = same, averaging over the alpha side.

This captures the interchain context: an amino acid that is globally enriched as a single mutant
but consistently *penalised* by its best-scoring beta partners will receive a lower marginal score.

In [ ]:
def compute_ml_marginals(pep, weight_by_score=True):
    pairs = _PAIRS_GRID.copy()
    pairs['predicted'] = ML_MODELS[pep].predict(_X_GRID)

    fx = SINGLE_FX[pep]
    if weight_by_score:
        alpha_w = fx[fx['chain']=='alpha'].set_index(['position','aa'])['log2fc'].clip(lower=0)
        beta_w  = fx[fx['chain']=='beta' ].set_index(['position','aa'])['log2fc'].clip(lower=0)
        pairs['w_alpha'] = (pairs.set_index(['alpha_pos','aa_alpha'])
                            .index.map(alpha_w).fillna(0).values)
        pairs['w_beta']  = (pairs.set_index(['beta_pos', 'aa_beta'])
                            .index.map(beta_w).fillna(0).values)
    else:
        pairs['w_alpha'] = pairs['w_beta'] = 1.0

    def _wm(grp, wcol):
        w = grp[wcol].values; v = grp['predicted'].values; s = w.sum()
        return (v * w).sum() / s if s > 0 else v.mean()

    alpha_rows = [
        {'chain': 'alpha', 'position': ap, 'aa': aa, 'ml_score': _wm(g, 'w_beta')}
        for (ap, aa), g in pairs.groupby(['alpha_pos','aa_alpha'], sort=False)
    ]
    beta_rows = [
        {'chain': 'beta', 'position': bp, 'aa': ba, 'ml_score': _wm(g, 'w_alpha')}
        for (bp, ba), g in pairs.groupby(['beta_pos','aa_beta'], sort=False)
    ]
    return pd.concat([pd.DataFrame(alpha_rows), pd.DataFrame(beta_rows)], ignore_index=True)


print('Computing ML marginals for all peptides...')
ML_MARGINALS = {pep: compute_ml_marginals(pep) for pep in PEPTIDES}
print('Done.')
print('\nTop-6 alpha marginals — DRG01:')
print(ML_MARGINALS['DRG01'].query("chain=='alpha'").nlargest(6,'ml_score')
      [['position','aa','ml_score']].to_string(index=False))
print('\nTop-6 beta marginals — DRG01:')
print(ML_MARGINALS['DRG01'].query("chain=='beta'").nlargest(6,'ml_score')
      [['position','aa','ml_score']].to_string(index=False))

## Diagnostics: ML Marginals vs. Original Single-Mutant Scores

In [ ]:
fig, axes = plt.subplots(2, len(PEPTIDES), figsize=(4 * len(PEPTIDES), 7), sharey='row')

for col, pep in enumerate(PEPTIDES):
    for row, chain in enumerate(['alpha', 'beta']):
        ax = axes[row, col]
        ml     = ML_MARGINALS[pep].query(f"chain=='{chain}'")
        orig   = SINGLE_FX[pep][SINGLE_FX[pep]['chain'] == chain]
        merged = ml.merge(orig[['position','aa','log2fc']], on=['position','aa'], how='inner')

        ax.scatter(merged['log2fc'], merged['ml_score'],
                   c=merged['position'], cmap='tab10', alpha=0.7, s=20, linewidths=0)
        for _, r in merged.nlargest(5, 'ml_score').iterrows():
            ax.annotate(f'{r["aa"]}{int(r["position"])}',
                        (r['log2fc'], r['ml_score']), fontsize=6.5,
                        xytext=(2, 2), textcoords='offset points')

        ax.axhline(0, color='#ccc', lw=0.7, ls='--')
        ax.axvline(0, color='#ccc', lw=0.7, ls='--')
        ax.set_xlabel('orig log2fc', fontsize=8)
        if col == 0:
            ax.set_ylabel(f'{chain} ML marginal', fontsize=8)
        ax.set_title(f'{pep} {chain}  R²={ML_R2[pep]:.2f}', fontsize=8)

plt.suptitle('ML marginal scores vs. original single-mutant log2fc\n(colour = CDR3 position)',
             fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'ml_marginals_vs_orig.pdf', bbox_inches='tight')
plt.show()

## Composite Score Function and Library Design

`pool_ml_composite_scores` is a drop-in replacement for the original `pool_composite_scores`.
It z-scores the ML marginals; no separate synergy bonus is needed because the model has
already learned the interchain context.

In [ ]:
def pool_ml_composite_scores(pep_list, chain):
    dfs = [ML_MARGINALS[p][ML_MARGINALS[p]['chain'] == chain][['position','aa','ml_score']]
           for p in pep_list if p in ML_MARGINALS]
    pooled = (pd.concat(dfs, ignore_index=True)
              .groupby(['position','aa'])['ml_score'].mean().reset_index()
              .rename(columns={'ml_score': 'base_score'}))
    mu    = pooled['base_score'].mean()
    sigma = pooled['base_score'].std()
    pooled['z_base']    = (pooled['base_score'] - mu) / (sigma if sigma > 0 else 1.0)
    pooled['composite'] = pooled['z_base']
    return pooled.sort_values(['position','composite'], ascending=[True, False])


print('DRG01 alpha — top-8 ML composite scores:')
print(pool_ml_composite_scores(['DRG01'], 'alpha').nlargest(8, 'composite').to_string(index=False))
print('\nDRG01 beta — top-8 ML composite scores:')
print(pool_ml_composite_scores(['DRG01'], 'beta').nlargest(8, 'composite').to_string(index=False))

In [ ]:
# ── Oligo construction helpers (copied from original notebook) ────────────

def _wt_triplets(wt_seq):
    t = {}
    for pos, aa in enumerate(wt_seq):
        for codon, enc_aa in GENETIC_CODE.items():
            if enc_aa == aa: t[pos] = codon; break
    return t


def _canonical_codon(pos, cands_by_pos, scores_by_pos, wt_aa=None, top_k=1):
    if pos not in cands_by_pos or not cands_by_pos[pos]:
        return None, frozenset(), 0.0
    top_aas   = [cands_by_pos[pos][i][0]
                 for i in range(min(top_k, len(cands_by_pos[pos])))]
    must_have = frozenset(top_aas) | (frozenset({wt_aa}) if wt_aa is not None else frozenset())
    pos_scores = scores_by_pos.get(pos, {})
    best_triplet, best_encoded, best_mean = None, frozenset(), -np.inf
    for triplet, props in IUPAC_PROPS.items():
        encoded = props['aas']
        if not must_have.issubset(encoded):
            continue
        mean_score = (
            sum(pos_scores.get(aa, 0.0) for aa in encoded)
            - STOP_PENALTY * props['stop_frac']
        ) / max(len(encoded), 1)
        if mean_score > best_mean:
            best_mean    = mean_score
            best_triplet = triplet
            best_encoded = encoded
    if best_triplet is None:
        return None, frozenset(), 0.0
    return best_triplet, frozenset(best_encoded), len(must_have) / len(best_encoded)


def _make_paired_oligo(oligo_id, oligo_type, alpha_enc, beta_enc,
                       wt_alpha, wt_beta, wt_tri_alpha, wt_tri_beta):
    alpha_len = len(wt_alpha)
    beta_len  = len(wt_beta)
    alpha_pos_aas = {p: frozenset({wt_alpha[p]}) for p in range(alpha_len)}
    beta_pos_aas  = {p: frozenset({wt_beta[p]})  for p in range(beta_len)}
    alpha_iupac   = dict(wt_tri_alpha)
    beta_iupac    = dict(wt_tri_beta)
    effs      = {}
    diversity = 1
    for pos, (triplet, encoded, eff) in alpha_enc.items():
        alpha_pos_aas[pos] = encoded; alpha_iupac[pos] = triplet
        effs[('alpha', pos)] = eff;   diversity *= len(encoded)
    for pos, (triplet, encoded, eff) in beta_enc.items():
        beta_pos_aas[pos] = encoded; beta_iupac[pos] = triplet
        effs[('beta', pos)] = eff;   diversity *= len(encoded)
    alpha_dna = ''.join(alpha_iupac[p] for p in range(alpha_len))
    beta_dna  = ''.join(beta_iupac[p]  for p in range(beta_len))
    return {
        'oligo_id':           oligo_id,
        'oligo_type':         oligo_type,
        'alpha_position_aas': alpha_pos_aas,
        'beta_position_aas':  beta_pos_aas,
        'alpha_iupac':        alpha_iupac,
        'beta_iupac':         beta_iupac,
        'alpha_dna':          alpha_dna,
        'beta_dna':           beta_dna,
        'paired_dna':         alpha_dna + beta_dna,
        'diversity':          diversity,
        'codon_efficiency':   effs,
        'is_leaky':           any(e < EFFICIENCY_THRESH for e in effs.values()),
        'alpha_varied_pos':   sorted(p for p, aas in alpha_pos_aas.items() if len(aas) > 1),
        'beta_varied_pos':    sorted(p for p, aas in beta_pos_aas.items()  if len(aas) > 1),
        'n_alpha_varied':     sum(1 for aas in alpha_pos_aas.values() if len(aas) > 1),
        'n_beta_varied':      sum(1 for aas in beta_pos_aas.values()  if len(aas) > 1),
    }


def design_paired_pool_ml(pep, wt_alpha, wt_beta, target_diversity, max_diversity):
    """Design a TCRAFT paired oPool using ML-derived composite scores."""
    wt_tri_alpha = _wt_triplets(wt_alpha)
    wt_tri_beta  = _wt_triplets(wt_beta)

    alpha_df = pool_ml_composite_scores([pep], 'alpha')
    beta_df  = pool_ml_composite_scores([pep], 'beta')

    scores_by_key = {}
    cands_by_key  = {}
    for chain, df in [('alpha', alpha_df), ('beta', beta_df)]:
        for _, row in df.iterrows():
            key = (chain, int(row['position']))
            scores_by_key.setdefault(key, {})[row['aa']] = row['composite']
        for pos, grp in df.groupby('position'):
            top = (grp[grp['composite'] >= MIN_SCORE]
                   .nlargest(K_MAX, 'composite')[['aa', 'composite']].values.tolist())
            if top:
                cands_by_key[(chain, int(pos))] = top

    canon = {}
    for (chain, pos), cands in cands_by_key.items():
        wt_seq = wt_alpha if chain == 'alpha' else wt_beta
        t, enc, eff = _canonical_codon(
            pos, {pos: cands}, {pos: scores_by_key[(chain, pos)]},
            wt_aa=wt_seq[pos], top_k=TOP_K
        )
        if t is not None:
            canon[(chain, pos)] = (t, enc, eff)

    sorted_pos = sorted(
        cands_by_key,
        key=lambda k: max(sc for _, sc in cands_by_key[k]),
        reverse=True
    )
    n_pos = len(sorted_pos)

    tier1, tier2, tier3 = [], [], []
    tier1_div    = 1
    tier1_budget = int(target_diversity * TIER1_FRAC)

    for key in sorted_pos:
        chain_k, pos_k = key
        if key not in canon:      tier3.append(key); continue
        if pos_k in TIER1_EXCLUDE.get(chain_k, set()): continue
        proposed = tier1_div * len(canon[key][1])
        if proposed > tier1_budget: continue
        tier1_div = proposed
        tier1.append(key)

    remaining = [k for k in sorted_pos if k not in set(tier1)]
    if TIER1_FRAC < 1.0:
        ext_budget = target_diversity - tier1_div
        for key in remaining:
            if key not in canon:      tier3.append(key); continue
            cost = tier1_div * (len(canon[key][1]) - 1)
            if cost > ext_budget:     continue
            if tier1_div + cost > max_diversity: continue
            tier2.append(key)
            ext_budget -= cost

    in_t1t2 = set(tier1 + tier2)
    tier3 += [k for k in sorted_pos if k not in in_t1t2 and k not in tier3]

    total_div = tier1_div + sum(
        tier1_div * (len(canon[k][1]) - 1) for k in tier2 if k in canon
    )

    t1a = sorted(p for c, p in tier1 if c == 'alpha')
    t1b = sorted(p for c, p in tier1 if c == 'beta')
    t2a = sorted(p for c, p in tier2 if c == 'alpha')
    t2b = sorted(p for c, p in tier2 if c == 'beta')
    print(f'  {n_pos} enriched positions across both chains  [TOP_K={TOP_K}]')
    print(f'  Tier 1 ({len(tier1)} pos):  alpha={t1a}  beta={t1b}  — backbone div = {tier1_div:,}')
    print(f'  Tier 2 ({len(tier2)} pos):  alpha_ext={t2a}  beta_ext={t2b}')
    print(f'  Tier 3 ({len(tier3)} pos):  {tier3}')
    print(f'  Estimated diversity: {total_div:,}  (target {target_diversity:,}  max {max_diversity:,})')

    def a_enc(keys):
        return {p: canon[('alpha', p)] for (c, p) in keys if c=='alpha' and ('alpha',p) in canon}
    def b_enc(keys):
        return {p: canon[('beta',  p)] for (c, p) in keys if c=='beta'  and ('beta', p) in canon}

    seen_dna = set()
    oligos   = []

    def add(oligo):
        if oligo['paired_dna'] not in seen_dna:
            seen_dna.add(oligo['paired_dna']); oligos.append(oligo); return True
        return False

    tier1_set = set(tier1)
    add(_make_paired_oligo(f'{pep}_backbone', 'tier1_backbone',
                           a_enc(tier1_set), b_enc(tier1_set),
                           wt_alpha, wt_beta, wt_tri_alpha, wt_tri_beta))

    for key in tier2:
        chain_k, pos_k = key
        ext_set = tier1_set | {key}
        label   = f'{"a" if chain_k=="alpha" else "b"}p{pos_k:02d}'
        add(_make_paired_oligo(f'{pep}_ext_{label}', 'tier2_ext',
                               a_enc(ext_set), b_enc(ext_set),
                               wt_alpha, wt_beta, wt_tri_alpha, wt_tri_beta))

    for o in oligos:
        o['unique_diversity'] = (o['diversity'] if o['oligo_type'] == 'tier1_backbone'
                                 else o['diversity'] - tier1_div)

    if total_div < target_diversity * 0.7:
        print(f'  NOTE: diversity {total_div:,} is below 70% of target {target_diversity:,}')
    print(f'  Total oligos: {len(oligos)}')
    return oligos


print(f'Designing {len(PEPTIDES)} ML-guided paired pools...')
POOLS_ML = {}
for pep in PEPTIDES:
    print(f'\n── {pep} ──')
    POOLS_ML[pep] = design_paired_pool_ml(
        pep, WT_ALPHA, WT_BETA, TARGET_DIVERSITY, MAX_DIVERSITY
    )

## Diversity Check

In [ ]:
print(f'{"Peptide":<8}  {"n_oligos":>9}  {"backbone_div":>14}  {"total_div":>11}  {"target":>11}  flag')
print('-' * 70)
for pep in PEPTIDES:
    pool     = POOLS_ML[pep]
    backbone = next((o for o in pool if o['oligo_type'] == 'tier1_backbone'), None)
    bd       = backbone['diversity'] if backbone else '-'
    total    = sum(o['unique_diversity'] for o in pool)
    flag     = 'OVER MAX!' if total > MAX_DIVERSITY else ('OK' if total <= TARGET_DIVERSITY * 1.1 else 'above target')
    types    = {}
    for o in pool: types[o['oligo_type']] = types.get(o['oligo_type'], 0) + 1
    print(f'{pep:<8}  {len(pool):>9}  {bd:>14,}  {total:>11,}  {TARGET_DIVERSITY:>11,}  {flag}  {types}')

## Comparison: ML vs. Original Heuristic Design

Re-run the heuristic (position-level synergy bonus) design to generate a side-by-side comparison.

In [ ]:
# ── Heuristic composite score (needed only for comparison) ───────────────
def synergy_bonus_for_chain(pep_list, chain, min_epi=0.2):
    all_epi = pd.concat([EPISTASIS[p] for p in pep_list if not EPISTASIS[p].empty], ignore_index=True)
    if chain == 'alpha': pos_col, aa_col = 'alpha_pos','aa_alpha'
    else:                pos_col, aa_col = 'beta_pos', 'aa_beta'
    opos_col = 'beta_pos' if chain == 'alpha' else 'alpha_pos'
    pooled = all_epi.groupby([pos_col, aa_col, opos_col])['epistasis'].mean().reset_index()
    syn = defaultdict(float)
    for _, row in pooled.iterrows():
        if row['epistasis'] > min_epi:
            syn[(int(row[pos_col]), row[aa_col])] += row['epistasis']
    return dict(syn)

def pool_composite_scores(pep_list, chain):
    chain_dfs = [SINGLE_FX[p][SINGLE_FX[p]['chain']==chain] for p in pep_list if p in SINGLE_FX]
    pooled = (pd.concat(chain_dfs, ignore_index=True)
              .groupby(['position','aa'])['log2fc'].mean().reset_index()
              .rename(columns={'log2fc':'base_score'}))
    syn_raw = synergy_bonus_for_chain(pep_list, chain)
    pooled['syn_raw'] = pooled.apply(lambda r: syn_raw.get((int(r['position']), r['aa']), 0.0), axis=1)
    pooled['z_base']  = (pooled['base_score'] - pooled['base_score'].mean()) / pooled['base_score'].std()
    syn_std = pooled['syn_raw'].std()
    pooled['z_syn']   = (pooled['syn_raw'] - pooled['syn_raw'].mean()) / (syn_std if syn_std > 0 else 1.0)
    pooled['composite'] = pooled['z_base'] + SYN_WEIGHT * pooled['z_syn']
    return pooled.sort_values(['position','composite'], ascending=[True,False])

def design_paired_pool_heuristic(pep, wt_alpha, wt_beta, target_diversity, max_diversity):
    """Heuristic design (same logic as original notebook) — used only for comparison."""
    wt_tri_alpha = _wt_triplets(wt_alpha)
    wt_tri_beta  = _wt_triplets(wt_beta)
    alpha_df = pool_composite_scores([pep], 'alpha')
    beta_df  = pool_composite_scores([pep], 'beta')
    scores_by_key = {}; cands_by_key = {}
    for chain, df in [('alpha', alpha_df), ('beta', beta_df)]:
        for _, row in df.iterrows():
            key = (chain, int(row['position']))
            scores_by_key.setdefault(key, {})[row['aa']] = row['composite']
        for pos, grp in df.groupby('position'):
            top = (grp[grp['composite'] >= MIN_SCORE]
                   .nlargest(K_MAX, 'composite')[['aa','composite']].values.tolist())
            if top: cands_by_key[(chain, int(pos))] = top
    canon = {}
    for (chain, pos), cands in cands_by_key.items():
        wt_seq = wt_alpha if chain == 'alpha' else wt_beta
        t, enc, eff = _canonical_codon(pos, {pos: cands}, {pos: scores_by_key[(chain,pos)]},
                                        wt_aa=wt_seq[pos], top_k=TOP_K)
        if t is not None: canon[(chain, pos)] = (t, enc, eff)
    sorted_pos = sorted(cands_by_key, key=lambda k: max(sc for _,sc in cands_by_key[k]), reverse=True)
    tier1, tier2, tier3 = [], [], []; tier1_div = 1
    tier1_budget = int(target_diversity * TIER1_FRAC)
    for key in sorted_pos:
        chain_k, pos_k = key
        if key not in canon:      tier3.append(key); continue
        if pos_k in TIER1_EXCLUDE.get(chain_k, set()): continue
        proposed = tier1_div * len(canon[key][1])
        if proposed > tier1_budget: continue
        tier1_div = proposed; tier1.append(key)
    remaining = [k for k in sorted_pos if k not in set(tier1)]
    if TIER1_FRAC < 1.0:
        ext_budget = target_diversity - tier1_div
        for key in remaining:
            if key not in canon:      tier3.append(key); continue
            cost = tier1_div * (len(canon[key][1]) - 1)
            if cost > ext_budget or tier1_div + cost > max_diversity: continue
            tier2.append(key); ext_budget -= cost
    in_t1t2 = set(tier1 + tier2)
    tier3 += [k for k in sorted_pos if k not in in_t1t2 and k not in tier3]
    def a_enc(keys):
        return {p: canon[('alpha',p)] for (c,p) in keys if c=='alpha' and ('alpha',p) in canon}
    def b_enc(keys):
        return {p: canon[('beta',p)]  for (c,p) in keys if c=='beta'  and ('beta',p)  in canon}
    seen_dna = set(); oligos = []
    def add(o):
        if o['paired_dna'] not in seen_dna: seen_dna.add(o['paired_dna']); oligos.append(o)
    tier1_set = set(tier1)
    add(_make_paired_oligo(f'{pep}_backbone','tier1_backbone',
                           a_enc(tier1_set),b_enc(tier1_set),
                           wt_alpha,wt_beta,wt_tri_alpha,wt_tri_beta))
    for key in tier2:
        chain_k,pos_k = key; ext_set = tier1_set | {key}
        label = f'{"a" if chain_k=="alpha" else "b"}p{pos_k:02d}'
        add(_make_paired_oligo(f'{pep}_ext_{label}','tier2_ext',
                               a_enc(ext_set),b_enc(ext_set),
                               wt_alpha,wt_beta,wt_tri_alpha,wt_tri_beta))
    for o in oligos:
        o['unique_diversity'] = (o['diversity'] if o['oligo_type']=='tier1_backbone'
                                 else o['diversity'] - tier1_div)
    return oligos

print('Running heuristic design for comparison...')
POOLS_HEURISTIC = {pep: design_paired_pool_heuristic(
    pep, WT_ALPHA, WT_BETA, TARGET_DIVERSITY, MAX_DIVERSITY) for pep in PEPTIDES}
print('Done.')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
print(f'{"Peptide":<8}  {"R²":>6}  {"H oligos":>9}  {"H div":>12}  {"ML oligos":>10}  {"ML div":>12}')
print('─' * 65)
for pep in PEPTIDES:
    hd = sum(o['unique_diversity'] for o in POOLS_HEURISTIC[pep])
    md = sum(o['unique_diversity'] for o in POOLS_ML[pep])
    print(f'{pep:<8}  {ML_R2[pep]:>6.3f}  {len(POOLS_HEURISTIC[pep]):>9}  {hd:>12,}  '
          f'{len(POOLS_ML[pep]):>10}  {md:>12,}')

# ── IUPAC codon diffs for highest-R² peptide ─────────────────────────────
ref_pep = max(PEPTIDES, key=lambda p: ML_R2[p])
print(f'\n── Changed backbone IUPAC codons — {ref_pep}  (R²={ML_R2[ref_pep]:.3f}) ──')
orig_bb = next(o for o in POOLS_HEURISTIC[ref_pep] if o['oligo_type'] == 'tier1_backbone')
ml_bb   = next(o for o in POOLS_ML[ref_pep]        if o['oligo_type'] == 'tier1_backbone')

diffs = []
for chain, iupac_key, wt_seq in [('alpha','alpha_iupac',WT_ALPHA),
                                   ('beta', 'beta_iupac', WT_BETA)]:
    for pos in range(len(wt_seq)):
        oc = orig_bb[iupac_key][pos]
        mc = ml_bb[iupac_key][pos]
        if oc != mc:
            o_aas = IUPAC_PROPS.get(oc, {}).get('aas', set())
            m_aas = IUPAC_PROPS.get(mc, {}).get('aas', set())
            diffs.append({'chain': chain, 'pos': pos, 'wt': wt_seq[pos],
                          'heuristic_codon': oc, 'heuristic_aas': ''.join(sorted(o_aas)),
                          'ml_codon': mc, 'ml_aas': ''.join(sorted(m_aas)),
                          'dropped': ''.join(sorted(o_aas - m_aas)),
                          'gained':  ''.join(sorted(m_aas - o_aas))})

if diffs:
    print(pd.DataFrame(diffs).to_string(index=False))
else:
    print('No backbone codon changes for this peptide.')

In [ ]:
# ── Heatmap: composite score comparison (alpha CDR3, ref peptide) ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (title, fn) in zip(axes, [
    ('Heuristic composite score', pool_composite_scores),
    ('ML composite score',        pool_ml_composite_scores),
]):
    df    = fn([ref_pep], 'alpha')
    pivot = (df.pivot_table(index='aa', columns='position', values='composite', aggfunc='mean')
               .reindex(index=list('ACDEFGHIKLMNPQRSTVWY')))
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_xticks(range(ALPHA_LEN))
    ax.set_xticklabels([f'{wt}α{i}' for i, wt in enumerate(WT_ALPHA)], fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_title(f'{title} — {ref_pep} α CDR3', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f'Composite score heatmap — heuristic vs. ML  ({ref_pep} α CDR3)', fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / f'ml_vs_heuristic_heatmap_{ref_pep}.pdf', bbox_inches='tight')
plt.show()

## Export ML-Guided oPool to CSV

In [ ]:
def pool_to_df(pep, pool):
    rows, cumul = [], 0
    for o in pool:
        cumul += o['unique_diversity']
        alpha_var = '; '.join(
            f'p{p}:[{",".join(sorted(o["alpha_position_aas"][p]))}]'
            for p in o['alpha_varied_pos']
        )
        beta_var = '; '.join(
            f'p{p}:[{",".join(sorted(o["beta_position_aas"][p]))}]'
            for p in o['beta_varied_pos']
        )
        effs = [e for e in o['codon_efficiency'].values() if e < 1.0]
        rows.append({
            'oligo_id':               o['oligo_id'],
            'oligo_type':             o['oligo_type'],
            'alpha_dna':              o['alpha_dna'],
            'beta_dna':               o['beta_dna'],
            'paired_dna':             o['paired_dna'],
            'paired_dna_length_nt':   len(o['paired_dna']),
            'n_alpha_varied':         o['n_alpha_varied'],
            'n_beta_varied':          o['n_beta_varied'],
            'alpha_varied_positions': alpha_var,
            'beta_varied_positions':  beta_var,
            'diversity':              o['diversity'],
            'unique_diversity':       o['unique_diversity'],
            'cumulative_diversity':   cumul,
            'mean_codon_efficiency':  round(np.mean(effs), 3) if effs else 1.0,
            'is_leaky':               o.get('is_leaky', False),
        })
    return pd.DataFrame(rows)


idt_rows = []
for pep, pool in POOLS_ML.items():
    df   = pool_to_df(pep, pool)
    path = OUT_DIR / f'opool_paired_ml_{pep}.csv'
    df.to_csv(path, index=False)
    print(f'Wrote {path}  ({len(df)} oligos)')
    display(df[['oligo_id','oligo_type','alpha_dna','beta_dna',
                'n_alpha_varied','n_beta_varied','diversity','unique_diversity']])
    order = df[['oligo_id', 'paired_dna']].copy()
    order.insert(0, 'Pool Name', pep)
    idt_rows.append(order)

idt_df   = pd.concat(idt_rows, ignore_index=True)
idt_df   = idt_df.rename(columns={'oligo_id': 'Sequence Name',
                                   'paired_dna': 'Sequence (no adapters)'})
idt_path = OUT_DIR / 'IDT_opool_paired_ml_order.csv'
idt_df.to_csv(idt_path, index=False)
print(f'\nIDT order sheet → {idt_path}  ({len(idt_df)} oligos total)')